In [1]:
"""
=============================================================================
Federated Learning: Server-Side Model Aggregation Using Weighted FedAvg
            and Global Model Distribution
=============================================================================
Dataset : diabetes.csv (Pima Indians Diabetes Dataset)
          Place diabetes.csv in the same folder as this script.
=============================================================================
"""

import os
import copy
import time
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import OrderedDict

# =============================================================================
# LOGGING SETUP
# =============================================================================
def setup_logging(log_file="federated_learning.log"):
    log_format  = "%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s"
    date_format = "%Y-%m-%d %H:%M:%S"
    logging.basicConfig(
        level=logging.INFO,
        format=log_format,
        datefmt=date_format,
        handlers=[
            logging.FileHandler(log_file, mode="w"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger("FL-Main")


# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # Dataset
    "data_path"          : "diabetes.csv",
    "target_column"      : "Outcome",
    "test_size"          : 0.2,

    # Federated Learning
    "num_clients"        : 5,
    "num_rounds"         : 15,
    "fraction_clients"   : 1.0,       # fraction of clients used per round

    # Local Training
    "local_epochs"       : 5,
    "local_batch_size"   : 32,
    "local_lr"           : 0.01,
    "local_momentum"     : 0.9,

    # Evaluation
    "test_batch_size"    : 256,

    # Data distribution
    "iid"                : True,      # True = IID,  False = Non-IID

    # Reproducibility
    "seed"               : 42,
}


# =============================================================================
# MODEL  —  MLP for tabular binary classification
# =============================================================================
class DiabetesMLP(nn.Module):
    """
    Multi-Layer Perceptron for Diabetes binary classification.
    Input  : (B, num_features)
    Output : (B, 2)   ->  class 0 = No Diabetes, class 1 = Diabetes
    """
    def __init__(self, input_dim: int):
        super(DiabetesMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)          # binary output
        )

    def forward(self, x):
        return self.network(x)


# =============================================================================
# DATA LOADING & DISTRIBUTION
# =============================================================================
class DataDistributor:
    """Loads diabetes.csv, preprocesses it, and splits across FL clients."""

    def __init__(self, config, logger):
        self.config    = config
        self.logger    = logger
        self.input_dim = None         # set after loading

    def load_dataset(self):
        path = self.config["data_path"]
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Dataset not found at '{path}'. "
                f"Please place diabetes.csv in the same folder as this script."
            )

        self.logger.info(f"Loading dataset from: {os.path.abspath(path)}")
        df = pd.read_csv(path)
        self.logger.info(f"  Shape            : {df.shape}")
        self.logger.info(f"  Columns          : {df.columns.tolist()}")
        self.logger.info(
            f"  Class distribution:\n"
            f"{df[self.config['target_column']].value_counts().to_string()}"
        )
        self.logger.info(f"  Missing values   : {df.isnull().sum().sum()}")

        target = self.config["target_column"]
        X = df.drop(columns=[target]).values.astype(np.float32)
        y = df[target].values.astype(np.int64)
        self.input_dim = X.shape[1]
        self.logger.info(f"  Feature dim      : {self.input_dim}")

        # Stratified train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.config["test_size"],
            random_state=self.config["seed"],
            stratify=y
        )

        # Standardise — fit only on train set
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)
        self.logger.info(
            f"  Train samples    : {len(X_train)} | "
            f"Test samples: {len(X_test)}"
        )

        train_dataset = TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long)
        )
        test_dataset = TensorDataset(
            torch.tensor(X_test, dtype=torch.float32),
            torch.tensor(y_test, dtype=torch.long)
        )
        return train_dataset, test_dataset

    # ------------------------------------------------------------------
    def distribute_iid(self, dataset, num_clients):
        """Randomly partition training data equally among clients (IID)."""
        self.logger.info("Data distribution mode: IID")
        indices = np.random.permutation(len(dataset))
        splits  = np.array_split(indices, num_clients)
        client_datasets = {}
        for cid, idx in enumerate(splits):
            client_datasets[cid] = Subset(dataset, idx.tolist())
            self.logger.info(f"  Client {cid:02d} -> {len(idx)} samples")
        return client_datasets

    def distribute_non_iid(self, dataset, num_clients):
        """
        Non-IID: sort by label -> split into shards -> assign 2 shards/client.
        Creates skewed label distributions across clients.
        """
        self.logger.info("Data distribution mode: Non-IID (shard-based)")
        labels     = np.array([dataset[i][1].item() for i in range(len(dataset))])
        sorted_idx = np.argsort(labels)
        num_shards = num_clients * 2
        shards     = np.array_split(sorted_idx, num_shards)
        shard_ids  = list(range(num_shards))
        np.random.shuffle(shard_ids)

        client_datasets = {}
        for cid in range(num_clients):
            assigned = shard_ids[cid * 2 : (cid + 1) * 2]
            idx      = np.concatenate([shards[s] for s in assigned]).tolist()
            client_datasets[cid] = Subset(dataset, idx)
            unique, counts = np.unique(labels[idx], return_counts=True)
            dist = dict(zip(unique.astype(int), counts))
            self.logger.info(
                f"  Client {cid:02d} -> {len(idx)} samples | "
                f"Label dist: {dist}"
            )
        return client_datasets

    def get_test_loader(self, test_dataset):
        return DataLoader(
            test_dataset,
            batch_size=self.config["test_batch_size"],
            shuffle=False
        )


# =============================================================================
# FEDERATED CLIENT
# =============================================================================
class FederatedClient:
    """
    Simulates one FL participant.
    Receives global weights -> trains locally -> returns (weights, n_samples).
    """

    def __init__(self, client_id, dataset, input_dim, config, device):
        self.client_id   = client_id
        self.dataset     = dataset
        self.input_dim   = input_dim
        self.config      = config
        self.device      = device
        self.logger      = logging.getLogger(f"Client-{client_id:02d}")
        self.num_samples = len(dataset)

    def train(self, global_weights):
        self.logger.info(
            f"Local training start | Samples: {self.num_samples} | "
            f"Epochs: {self.config['local_epochs']}"
        )

        # Initialise local model with global weights
        model = DiabetesMLP(self.input_dim).to(self.device)
        model.load_state_dict(global_weights)
        model.train()

        loader = DataLoader(
            self.dataset,
            batch_size=self.config["local_batch_size"],
            shuffle=True,
            drop_last=False
        )
        optimizer = optim.SGD(
            model.parameters(),
            lr=self.config["local_lr"],
            momentum=self.config["local_momentum"],
            weight_decay=1e-4
        )
        criterion = nn.CrossEntropyLoss()

        total_loss, total_correct, total_samples = 0.0, 0, 0

        for epoch in range(self.config["local_epochs"]):
            epoch_loss, epoch_correct, epoch_samples = 0.0, 0, 0
            for data, target in loader:
                data, target = data.to(self.device), target.to(self.device)
                optimizer.zero_grad()
                output = model(data)
                loss   = criterion(output, target)
                loss.backward()
                optimizer.step()

                bs             = len(data)
                epoch_loss    += loss.item() * bs
                epoch_correct += output.argmax(1).eq(target).sum().item()
                epoch_samples += bs

            avg_loss = epoch_loss / epoch_samples
            acc      = 100.0 * epoch_correct / epoch_samples
            total_loss    += epoch_loss
            total_correct += epoch_correct
            total_samples += epoch_samples
            self.logger.info(
                f"  Epoch [{epoch+1}/{self.config['local_epochs']}] "
                f"Loss: {avg_loss:.4f} | Acc: {acc:.2f}%"
            )

        final_loss = total_loss / total_samples
        final_acc  = 100.0 * total_correct / total_samples
        self.logger.info(
            f"Local training done | Avg Loss: {final_loss:.4f} | "
            f"Avg Acc: {final_acc:.2f}%"
        )
        return model.state_dict(), self.num_samples


# =============================================================================
# FEDERATED SERVER  —  Weighted FedAvg + Global Distribution
# =============================================================================
class FederatedServer:
    """
    Central FL server.
      - Selects clients each round
      - Distributes global model weights  (Global Model Distribution)
      - Aggregates updates via Weighted FedAvg  (Server-Side Aggregation)
      - Evaluates global model on held-out test set
    """

    def __init__(self, input_dim, config, test_loader, device):
        self.config      = config
        self.test_loader = test_loader
        self.device      = device
        self.logger      = logging.getLogger("FL-Server")

        self.global_model = DiabetesMLP(input_dim).to(device)
        self.logger.info(
            "Global model (DiabetesMLP) initialised with random weights."
        )
        self._log_model_info()

    def _log_model_info(self):
        total     = sum(p.numel() for p in self.global_model.parameters())
        trainable = sum(
            p.numel() for p in self.global_model.parameters() if p.requires_grad
        )
        self.logger.info(
            f"Model params -> Total: {total:,} | Trainable: {trainable:,}"
        )

    # ------------------------------------------------------------------
    def get_global_weights(self):
        return copy.deepcopy(self.global_model.state_dict())

    def select_clients(self, all_clients):
        k        = max(1, int(len(all_clients) * self.config["fraction_clients"]))
        selected = np.random.choice(all_clients, k, replace=False).tolist()
        self.logger.info(
            f"Selected {k}/{len(all_clients)} clients: "
            f"{[c.client_id for c in selected]}"
        )
        return selected

    def distribute_global_model(self, clients):
        """
        ── Global Model Distribution ──────────────────────────────────
        Server sends the current global weights to every selected client.
        In real FL this would be an encrypted network transfer; here we
        return a deep-copy of the state dict.
        """
        weights = self.get_global_weights()
        self.logger.info(
            f"[Distribution] Global model sent to clients: "
            f"{[c.client_id for c in clients]}"
        )
        return weights

    def weighted_fedavg(self, client_updates):
        """
        ── Weighted Federated Averaging (FedAvg) ──────────────────────
        McMahan et al., 2017  —  "Communication-Efficient Learning of
        Deep Networks from Decentralized Data"

            w_global ← Σ_k  (n_k / N) × w_k

        n_k : local sample count for client k
        N   : total samples across all selected clients
        w_k : local model weights after training
        """
        self.logger.info(
            "[Aggregation] Weighted FedAvg — computing new global weights..."
        )
        total_samples = sum(n for _, n in client_updates)
        self.logger.info(f"  Total samples contributed: {total_samples}")

        # Initialise accumulator
        agg       = OrderedDict()
        ref_state = client_updates[0][0]
        for key in ref_state:
            agg[key] = torch.zeros_like(ref_state[key], dtype=torch.float32)

        # Weighted sum
        for idx, (state_dict, n_samples) in enumerate(client_updates):
            weight = n_samples / total_samples
            self.logger.info(
                f"  Client [{idx}] | n_k={n_samples} | weight={weight:.4f}"
            )
            for key in agg:
                agg[key] += weight * state_dict[key].float()

        self.logger.info("[Aggregation] Complete.")
        return agg

    def update_global_model(self, aggregated_weights):
        self.global_model.load_state_dict(aggregated_weights)
        self.logger.info("Global model weights updated with aggregated result.")

    def evaluate(self):
        """Evaluate global model on the centralised test set."""
        self.global_model.eval()
        criterion = nn.CrossEntropyLoss()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels      = [], []

        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.global_model(data)
                loss   = criterion(output, target)
                preds  = output.argmax(dim=1)

                total_loss  += loss.item() * len(data)
                correct     += preds.eq(target).sum().item()
                total       += len(data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(target.cpu().numpy())

        avg_loss = total_loss / total
        accuracy = 100.0 * correct / total

        all_preds  = np.array(all_preds)
        all_labels = np.array(all_labels)

        # Per-class accuracy
        for cls in np.unique(all_labels):
            mask    = all_labels == cls
            cls_acc = 100.0 * (all_preds[mask] == cls).sum() / mask.sum()
            label   = "No Diabetes" if cls == 0 else "Diabetes   "
            self.logger.info(
                f"  Class {cls} ({label}) Accuracy: {cls_acc:.2f}%  "
                f"[{mask.sum()} samples]"
            )

        self.logger.info(
            f"  Overall -> Loss: {avg_loss:.4f} | "
            f"Accuracy: {accuracy:.2f}% ({correct}/{total})"
        )
        return avg_loss, accuracy


# =============================================================================
# FL RUNNER  —  Orchestrator
# =============================================================================
class FederatedLearningRunner:
    """End-to-end Federated Learning pipeline orchestrator."""

    def __init__(self, config):
        self.config = config
        self.logger = logging.getLogger("FL-Runner")
        self.device = self._get_device()
        self._set_seed()

    def _get_device(self):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.logger.info(f"Device: {device}")
        return device

    def _set_seed(self):
        torch.manual_seed(self.config["seed"])
        np.random.seed(self.config["seed"])
        self.logger.info(f"Random seed: {self.config['seed']}")

    # ------------------------------------------------------------------
    def run(self):
        self.logger.info("=" * 70)
        self.logger.info(
            "  Federated Learning — Weighted FedAvg — Diabetes Dataset"
        )
        self.logger.info("=" * 70)

        # ── 1. Data ────────────────────────────────────────────────────
        distributor            = DataDistributor(self.config, self.logger)
        train_data, test_data  = distributor.load_dataset()
        input_dim              = distributor.input_dim

        if self.config["iid"]:
            client_datasets = distributor.distribute_iid(
                train_data, self.config["num_clients"]
            )
        else:
            client_datasets = distributor.distribute_non_iid(
                train_data, self.config["num_clients"]
            )

        test_loader = distributor.get_test_loader(test_data)

        # ── 2. Clients ─────────────────────────────────────────────────
        clients = [
            FederatedClient(cid, ds, input_dim, self.config, self.device)
            for cid, ds in client_datasets.items()
        ]
        self.logger.info(f"Initialised {len(clients)} federated clients.")

        # ── 3. Server ──────────────────────────────────────────────────
        server = FederatedServer(input_dim, self.config, test_loader, self.device)

        # ── 4. Pre-training baseline ───────────────────────────────────
        self.logger.info("\n[PRE-TRAINING EVALUATION — random weights]")
        init_loss, init_acc = server.evaluate()

        history = {
            "round"    : [0],
            "test_loss": [init_loss],
            "test_acc" : [init_acc],
        }

        # ── 5. FL Communication Rounds ─────────────────────────────────
        for rnd in range(1, self.config["num_rounds"] + 1):
            self.logger.info("\n" + "─" * 70)
            self.logger.info(f"  ROUND {rnd}/{self.config['num_rounds']}")
            self.logger.info("─" * 70)
            t0 = time.time()

            # a) Client selection
            selected = server.select_clients(clients)

            # b) Global Model Distribution  ←──────────────────────────
            global_weights = server.distribute_global_model(selected)

            # c) Local training
            client_updates = []
            for client in selected:
                w, n = client.train(global_weights)
                client_updates.append((w, n))

            # d) Weighted FedAvg aggregation  ←────────────────────────
            agg_weights = server.weighted_fedavg(client_updates)

            # e) Update global model
            server.update_global_model(agg_weights)

            # f) Evaluate
            self.logger.info(f"\n[ROUND {rnd} — GLOBAL MODEL EVALUATION]")
            loss, acc = server.evaluate()
            elapsed   = time.time() - t0

            self.logger.info(
                f"ROUND {rnd} DONE | Time: {elapsed:.1f}s | "
                f"Loss: {loss:.4f} | Accuracy: {acc:.2f}%"
            )

            history["round"].append(rnd)
            history["test_loss"].append(loss)
            history["test_acc"].append(acc)

        # ── 6. Final Summary Table ─────────────────────────────────────
        self.logger.info("\n" + "=" * 70)
        self.logger.info("TRAINING COMPLETE — ROUND-WISE SUMMARY")
        self.logger.info("=" * 70)
        self.logger.info(f"{'Round':<10} {'Test Loss':<14} {'Test Accuracy'}")
        self.logger.info("-" * 40)
        for i, rnd in enumerate(history["round"]):
            tag = "(init)" if rnd == 0 else ""
            self.logger.info(
                f"{str(rnd)+' '+tag:<14} "
                f"{history['test_loss'][i]:<14.4f} "
                f"{history['test_acc'][i]:.2f}%"
            )

        best_idx = int(np.argmax(history["test_acc"][1:])) + 1
        best_acc = history["test_acc"][best_idx]
        best_rnd = history["round"][best_idx]

        self.logger.info("=" * 70)
        self.logger.info(f"Best Accuracy  : {best_acc:.2f}%  at Round {best_rnd}")
        self.logger.info(
            f"Gain over init : +{best_acc - history['test_acc'][0]:.2f}%"
        )
        self.logger.info("=" * 70)
        self.logger.info("Full log saved to: federated_learning.log")

        return history


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    main_logger = setup_logging("federated_learning.log")
    main_logger.info("Starting Federated Learning — Diabetes Dataset")

    runner  = FederatedLearningRunner(CONFIG)
    history = runner.run()

2026-02-26 15:37:59 | INFO     | FL-Main              | Federated Learning Assignment - Weighted FedAvg
2026-02-26 15:37:59 | INFO     | FL-Runner            | Using device: cpu
2026-02-26 15:37:59 | INFO     | FL-Runner            | Random seed set to 42
2026-02-26 15:37:59 | INFO     | FL-Runner            | ======================================================================
2026-02-26 15:37:59 | INFO     | FL-Runner            |    FEDERATED LEARNING: Weighted FedAvg + Global Distribution
2026-02-26 15:37:59 | INFO     | FL-Runner            | ======================================================================
2026-02-26 15:37:59 | INFO     | FL-Runner            | Config: {'num_clients': 5, 'num_rounds': 10, 'fraction_clients': 1.0, 'local_epochs': 2, 'local_batch_size': 64, 'local_lr': 0.01, 'local_momentum': 0.9, 'dataset': 'MNIST', 'data_dir': './data', 'iid': True, 'test_batch_size': 256, 'seed': 42, 'shards_per_client': 2}
2026-02-26 15:37:59 | INFO     | FL-Runner      